# XLM-R Baseline — Sinhala Offensive Language Classification (with Bootstrap CI)

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/datasets/najathmohamed/sinhala-toxity-dataset/sentence-level-annotation (1).csv


In [2]:
!pip install -q --upgrade "transformers>=5.0.0" accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 104.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 79.6 MB/s eta 0:00:00:00:01


In [3]:
# ============================================================================
# DATA PREPARATION (Sinhala) — same split/labels as the Llama+LaBSE notebook
# ============================================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import gc
import os

from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

DATA_PATH = ("/kaggle/input/datasets/najathmohamed/sinhala-toxity-dataset/sentence-level-annotation (1).csv")

XLMR_MODEL = "xlm-roberta-base"

RANDOM_SEED = 42
TEST_SIZE = 0.1
VAL_SIZE = 0.1

BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64

MAX_LEN = 128
EPOCHS = 5
LR = 2e-5
WEIGHT_DECAY = 0.01
COMPUTE_DTYPE = torch.bfloat16

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

df = pd.read_csv(DATA_PATH)

label_map = {"Not offensive": 0, "Abusive": 1, "Hate-Inducing": 1}
df["label"] = df["Hate_speech"].map(label_map)
df = df.dropna(subset=["label"]).copy()
df["label"] = df["label"].astype(int)

sentences = df["Sentence"].astype(str).tolist()
labels = df["label"].values

print(f"Total examples: {len(sentences)}")
print(f"Class distribution: {np.bincount(labels)}")

X_train, X_test, y_train, y_test = train_test_split(
    sentences, labels, test_size=TEST_SIZE,
    random_state=RANDOM_SEED, stratify=labels,
)

train_idx, val_idx = train_test_split(
    np.arange(len(y_train)), test_size=VAL_SIZE,
    stratify=y_train, random_state=RANDOM_SEED,
)

X_train_arr = np.array(X_train, dtype=object)
X_adapter_train = X_train_arr[train_idx].tolist()
X_adapter_val = X_train_arr[val_idx].tolist()
y_adapter_train = y_train[train_idx]
y_adapter_val = y_train[val_idx]

print(f"Train: {len(X_adapter_train)}  Val: {len(X_adapter_val)}  Test: {len(X_test)}")

class_counts = np.bincount(y_adapter_train)
print(f"Train class counts: {class_counts}")


Total examples: 13518
Class distribution: [12262  1256]
Train: 10949  Val: 1217  Test: 1352
Train class counts: [9932 1017]


In [4]:
# ============================================================================
# LOAD XLM-R MODEL + TOKENIZER
# ============================================================================

from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("Loading XLM-R tokenizer + model...")

tokenizer = AutoTokenizer.from_pretrained(XLMR_MODEL)

model = AutoModelForSequenceClassification.from_pretrained(
    XLMR_MODEL, num_labels=2,
)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

print("Model loaded:", type(model).__name__)
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


Loading XLM-R tokenizer + model...


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: XLMRobertaForSequenceClassification
Total params: 278,045,186
Trainable params: 278,045,186
GPU memory allocated: 1.11 GB


In [5]:
# ============================================================================
# DATASET / COLLATE
# ============================================================================

class ToxicityDataset(Dataset):
    def __init__(self, sentences, labels, tokenizer, max_len=128):
        self.sentences = sentences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.sentences[idx],
            truncation=True,
            max_length=self.max_len,
        )
        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "label": int(self.labels[idx]),
        }


def collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    pad_id = tokenizer.pad_token_id

    input_ids, attention_mask, label_ids = [], [], []

    for x in batch:
        pad_len = max_len - len(x["input_ids"])
        input_ids.append(x["input_ids"] + [pad_id] * pad_len)
        attention_mask.append(x["attention_mask"] + [0] * pad_len)
        label_ids.append(x["label"])

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(label_ids, dtype=torch.long),
    }


train_dataset = ToxicityDataset(X_adapter_train, y_adapter_train, tokenizer, MAX_LEN)
val_dataset = ToxicityDataset(X_adapter_val, y_adapter_val, tokenizer, MAX_LEN)
test_dataset = ToxicityDataset(X_test, y_test, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

class_weights_tensor = torch.tensor(
    [1.0 / class_counts[0], 1.0 / class_counts[1]], dtype=torch.float32, device=model.device,
)
class_weights_tensor = class_weights_tensor / class_weights_tensor.sum()
print("Class weights:", class_weights_tensor)


Class weights: tensor([0.0929, 0.9071], device='cuda:0')


In [6]:
# ============================================================================
# FORWARD PASS HELPER
# ============================================================================

def run_batch(batch):
    input_ids = batch["input_ids"].to(model.device)
    attention_mask = batch["attention_mask"].to(model.device)
    labels = batch["labels"].to(model.device)

    with torch.autocast(device_type="cuda", dtype=COMPUTE_DTYPE):
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

    loss = F.cross_entropy(logits.float(), labels, weight=class_weights_tensor)
    return loss, logits


In [7]:
# ============================================================================
# OPTIMIZER
# ============================================================================

from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


Trainable params: 278,045,186


In [8]:
# ============================================================================
# TRAINING LOOP (resumable — skips epochs already checkpointed on disk)
# ============================================================================

CKPT_DIR = "/kaggle/working/checkpoints_xlmr"
os.makedirs(CKPT_DIR, exist_ok=True)

STEP_CKPT_EVERY = 200
PATIENCE = 3

def build_state():
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

def save_state(state, path):
    torch.save(state, path)

def restore_checkpoint(state):
    model.load_state_dict({k: v.to(model.device) for k, v in state.items()})

@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, all_preds, all_targets = 0.0, [], []
    for batch in loader:
        loss, logits = run_batch(batch)
        total_loss += loss.item()
        preds = logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(batch["labels"].numpy())
    avg_loss = total_loss / len(loader)
    return avg_loss, accuracy_score(all_targets, all_preds), f1_score(all_targets, all_preds)


best_val_loss = np.inf
best_state = None
epochs_no_improve = 0

for epoch in range(EPOCHS):
    epoch_num = epoch + 1
    ckpt_path = os.path.join(CKPT_DIR, f"epoch{epoch_num}_checkpoint.pt")

    # ---- SKIP if this epoch was already completed and saved ----
    if os.path.exists(ckpt_path):
        print(f"Epoch {epoch_num} checkpoint already exists — loading instead of retraining.")
        state = torch.load(ckpt_path)
        restore_checkpoint(state)
        val_loss, val_acc, val_f1 = evaluate(val_loader)
        print(f"Epoch {epoch_num} (loaded) | Val loss: {val_loss:.4f} | Val acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = build_state()
            save_state(best_state, os.path.join(CKPT_DIR, "best_checkpoint.pt"))
        continue

    # ---- TRAIN this epoch for real ----
    model.train()
    total_train_loss = 0.0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch_num}/{EPOCHS} [train]", unit="batch")
    for step, batch in enumerate(train_bar):
        optimizer.zero_grad(set_to_none=True)
        loss, _ = run_batch(batch)
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        train_bar.set_postfix(loss=f"{loss.item():.4f}")

        if (step + 1) % STEP_CKPT_EVERY == 0:
            save_state(build_state(), os.path.join(CKPT_DIR, "latest_step_checkpoint.pt"))

        del loss

    train_loss = total_train_loss / len(train_loader)
    gc.collect(); torch.cuda.empty_cache()

    val_loss, val_acc, val_f1 = evaluate(val_loader)
    gc.collect(); torch.cuda.empty_cache()

    print(f"\nEpoch {epoch_num}/{EPOCHS} | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | Val acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

    epoch_state = build_state()
    save_state(epoch_state, ckpt_path)
    print(f"Saved epoch {epoch_num} checkpoint -> {ckpt_path}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = epoch_state
        save_state(best_state, os.path.join(CKPT_DIR, "best_checkpoint.pt"))
        epochs_no_improve = 0
        print(f"\u2713 New best validation loss: {best_val_loss:.4f}")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print("Early stopping.")
            break


Epoch 1/5 [train]:   0%|          | 0/343 [00:00<?, ?batch/s]


Epoch 1/5 | Train loss: 0.6024 | Val loss: 0.4781 | Val acc: 0.8357 | Val F1: 0.4253
Saved epoch 1 checkpoint -> /kaggle/working/checkpoints_xlmr/epoch1_checkpoint.pt
✓ New best validation loss: 0.4781


Epoch 2/5 [train]:   0%|          | 0/343 [00:00<?, ?batch/s]


Epoch 2/5 | Train loss: 0.5170 | Val loss: 0.4637 | Val acc: 0.7149 | Val F1: 0.3656
Saved epoch 2 checkpoint -> /kaggle/working/checkpoints_xlmr/epoch2_checkpoint.pt
✓ New best validation loss: 0.4637


Epoch 3/5 [train]:   0%|          | 0/343 [00:00<?, ?batch/s]


Epoch 3/5 | Train loss: 0.4140 | Val loss: 0.4234 | Val acc: 0.9006 | Val F1: 0.5468
Saved epoch 3 checkpoint -> /kaggle/working/checkpoints_xlmr/epoch3_checkpoint.pt
✓ New best validation loss: 0.4234


Epoch 4/5 [train]:   0%|          | 0/343 [00:00<?, ?batch/s]


Epoch 4/5 | Train loss: 0.3515 | Val loss: 0.3515 | Val acc: 0.8932 | Val F1: 0.5667
Saved epoch 4 checkpoint -> /kaggle/working/checkpoints_xlmr/epoch4_checkpoint.pt
✓ New best validation loss: 0.3515


Epoch 5/5 [train]:   0%|          | 0/343 [00:00<?, ?batch/s]


Epoch 5/5 | Train loss: 0.2800 | Val loss: 0.4365 | Val acc: 0.8940 | Val F1: 0.5343
Saved epoch 5 checkpoint -> /kaggle/working/checkpoints_xlmr/epoch5_checkpoint.pt


In [9]:
# ============================================================================
# FULL TEST RESULTS PER EPOCH (with classification report)
# ============================================================================

@torch.no_grad()
def full_test_evaluation(loader, y_true):
    model.eval()
    all_preds = []
    for batch in tqdm(loader, desc="Evaluating", unit="batch"):
        input_ids = batch["input_ids"].to(model.device)
        attention_mask = batch["attention_mask"].to(model.device)
        with torch.autocast(device_type="cuda", dtype=COMPUTE_DTYPE):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
        preds = logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy())

    all_preds = np.array(all_preds)
    acc = accuracy_score(y_true, all_preds)
    f1 = f1_score(y_true, all_preds)
    report = classification_report(y_true, all_preds, target_names=["Non-toxic", "Toxic"])
    return acc, f1, report


for epoch_num in range(1, EPOCHS + 1):
    ckpt_path = os.path.join(CKPT_DIR, f"epoch{epoch_num}_checkpoint.pt")
    if not os.path.exists(ckpt_path):
        print(f"Epoch {epoch_num} checkpoint not found — skipping (training may have stopped early).")
        continue

    print("\n" + "=" * 80)
    print(f"EPOCH {epoch_num} — TEST RESULTS")
    print("=" * 80)

    state = torch.load(ckpt_path)
    restore_checkpoint(state)

    acc, f1, report = full_test_evaluation(test_loader, y_test)

    print(f"Accuracy: {acc:.4f}")
    print(f"F1 (Toxic/Offensive): {f1:.4f}")
    print("\nClassification report:")
    print(report)



EPOCH 1 — TEST RESULTS


Evaluating:   0%|          | 0/22 [00:00<?, ?batch/s]

Accuracy: 0.8410
F1 (Toxic/Offensive): 0.4297

Classification report:
              precision    recall  f1-score   support

   Non-toxic       0.96      0.86      0.91      1226
       Toxic       0.32      0.64      0.43       126

    accuracy                           0.84      1352
   macro avg       0.64      0.75      0.67      1352
weighted avg       0.90      0.84      0.86      1352


EPOCH 2 — TEST RESULTS


Evaluating:   0%|          | 0/22 [00:00<?, ?batch/s]

Accuracy: 0.7249
F1 (Toxic/Offensive): 0.3779

Classification report:
              precision    recall  f1-score   support

   Non-toxic       0.99      0.71      0.82      1226
       Toxic       0.24      0.90      0.38       126

    accuracy                           0.72      1352
   macro avg       0.61      0.80      0.60      1352
weighted avg       0.92      0.72      0.78      1352


EPOCH 3 — TEST RESULTS


...

Accuracy: 0.8928
F1 (Toxic/Offensive): 0.5215

Classification report:
              precision    recall  f1-score   support

   Non-toxic       0.96      0.92      0.94      1226
       Toxic       0.45      0.63      0.52       126

    accuracy                           0.89      1352
   macro avg       0.70      0.77      0.73      1352
weighted avg       0.91      0.89      0.90      1352


EPOCH 4 — TEST RESULTS


Evaluating:   0%|          | 0/22 [00:00<?, ?batch/s]

Accuracy: 0.8891
F1 (Toxic/Offensive): 0.5482

Classification report:
              precision    recall  f1-score   support

   Non-toxic       0.97      0.91      0.94      1226
       Toxic       0.44      0.72      0.55       126

    accuracy                           0.89      1352
   macro avg       0.71      0.81      0.74      1352
weighted avg       0.92      0.89      0.90      1352


EPOCH 5 — TEST RESULTS


Evaluating:   0%|          | 0/22 [00:00<?, ?batch/s]

Accuracy: 0.8972
F1 (Toxic/Offensive): 0.5531

Classification report:
              precision    recall  f1-score   support

   Non-toxic       0.97      0.92      0.94      1226
       Toxic       0.46      0.68      0.55       126

    accuracy                           0.90      1352
   macro avg       0.72      0.80      0.75      1352
weighted avg       0.92      0.90      0.91      1352



In [10]:
# ============================================================================
# FINAL TEST EVALUATION (best checkpoint) — LABELS / PREDICTIONS / PROBABILITIES
# ============================================================================
# Restores the best checkpoint (lowest validation loss) and runs one full
# forward pass over the test set, collecting true labels, argmax predictions,
# and the softmax probability of the "offensive" class. These three arrays
# feed directly into the bootstrap CI section below -- no extra forward
# passes through the model are needed there.

restore_checkpoint(best_state)
model.eval()

test_labels = []
test_predictions = []
test_probabilities = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Final test evaluation", unit="batch"):
        input_ids = batch["input_ids"].to(model.device)
        attention_mask = batch["attention_mask"].to(model.device)
        labels = batch["labels"]

        with torch.autocast(device_type="cuda", dtype=COMPUTE_DTYPE):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

        probs = F.softmax(logits.float(), dim=-1)[:, 1]
        preds = logits.argmax(dim=-1)

        test_labels.extend(labels.numpy())
        test_predictions.extend(preds.cpu().numpy())
        test_probabilities.extend(probs.cpu().numpy())

test_labels = np.array(test_labels)
test_predictions = np.array(test_predictions)
test_probabilities = np.array(test_probabilities)

print("\n" + "=" * 70)
print("FINAL TEST RESULTS (best checkpoint)")
print("=" * 70)
print(f"Accuracy: {accuracy_score(test_labels, test_predictions):.4f}")
print(f"F1 (Offensive): {f1_score(test_labels, test_predictions):.4f}")
print("\nClassification report:")
print(classification_report(test_labels, test_predictions, target_names=["Not offensive", "Offensive"]))


Final test evaluation:   0%|          | 0/22 [00:00<?, ?batch/s]


FINAL TEST RESULTS (best checkpoint)
Accuracy: 0.8891
F1 (Offensive): 0.5482

Classification report:
               precision    recall  f1-score   support

Not offensive       0.97      0.91      0.94      1226
    Offensive       0.44      0.72      0.55       126

     accuracy                           0.89      1352
    macro avg       0.71      0.81      0.74      1352
 weighted avg       0.92      0.89      0.90      1352



In [11]:
# ============================================================================
# SAVE — XLM-R BASELINE (best checkpoint), SINHALA
# ============================================================================

SAVE_DIR = "/kaggle/working/sinhala_xlmr_baseline"
os.makedirs(SAVE_DIR, exist_ok=True)

restore_checkpoint(best_state)

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

torch.save({"best_val_loss": best_val_loss, "language": "Sinhala", "model": XLMR_MODEL}, os.path.join(SAVE_DIR, "run_info.pt"))

print("\nSaved to:")
print(SAVE_DIR)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved to:
/kaggle/working/sinhala_xlmr_baseline


In [12]:
# ============================================================================
# BOOTSTRAP CLASSIFICATION REPORT (same method as the Llama/Gemma notebooks)
# ============================================================================
# A nonparametric bootstrap over already-computed test predictions:
# resample (y_true, y_pred[, y_score]) WITH replacement, same size as the
# test set, N_BOOTSTRAP times, recompute every metric that normally appears
# in classification_report on each resample, then report point estimate +
# bootstrap mean +/- std + 95% percentile CI.

from sklearn.metrics import (
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
)

N_BOOTSTRAP = 10000

BOOTSTRAP_SEED = 42

_CLS_METRIC_KEYS = [
    "precision_0", "recall_0", "f1_0", "support_0",
    "precision_1", "recall_1", "f1_1", "support_1",
    "accuracy",
    "macro_precision", "macro_recall", "macro_f1",
    "weighted_precision", "weighted_recall", "weighted_f1",
]


def _classification_report_row(y_true, y_pred):
    """One full set of classification-report-style metrics."""

    p, r, f, s = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1], average=None, zero_division=0
    )

    acc = accuracy_score(y_true, y_pred)

    pm, rm, fm, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1], average="macro", zero_division=0
    )

    pw, rw, fw, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1], average="weighted", zero_division=0
    )

    return {
        "precision_0": p[0], "recall_0": r[0], "f1_0": f[0], "support_0": s[0],
        "precision_1": p[1], "recall_1": r[1], "f1_1": f[1], "support_1": s[1],
        "accuracy": acc,
        "macro_precision": pm, "macro_recall": rm, "macro_f1": fm,
        "weighted_precision": pw, "weighted_recall": rw, "weighted_f1": fw,
    }


def bootstrap_classification_report(
    y_true,
    y_pred,
    y_score=None,
    n_boot=N_BOOTSTRAP,
    seed=BOOTSTRAP_SEED,
):
    """
    Nonparametric bootstrap over the test set.

    Resamples (y_true, y_pred[, y_score]) WITH replacement n_boot times
    (same size as the original test set each time), recomputes the full
    classification-report metric set on every resample (plus ROC-AUC /
    PR-AUC when y_score is given), and returns:
      - point:   metrics computed once on the full (un-resampled) test set
      - dists:   dict of key -> np.array of length n_boot
      - mean:    dict of key -> bootstrap mean
      - std:     dict of key -> bootstrap standard deviation
      - ci_lo/ci_hi: dict of key -> 2.5th / 97.5th percentile (95% CI)
    """

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    n = len(y_true)
    assert n == len(y_pred), "y_true and y_pred must be the same length"

    use_scores = y_score is not None

    if use_scores:
        y_score = np.asarray(y_score)
        assert n == len(y_score), "y_true and y_score must be the same length"

    keys = list(_CLS_METRIC_KEYS)
    if use_scores:
        keys += ["roc_auc", "pr_auc"]

    rng = np.random.default_rng(seed)

    dists = {k: np.empty(n_boot, dtype=np.float64) for k in keys}

    for i in range(n_boot):

        idx = rng.integers(0, n, size=n)

        yt = y_true[idx]
        yp = y_pred[idx]

        row = _classification_report_row(yt, yp)

        for k in _CLS_METRIC_KEYS:
            dists[k][i] = row[k]

        if use_scores:

            ys = y_score[idx]

            try:
                dists["roc_auc"][i] = roc_auc_score(yt, ys)
            except ValueError:
                dists["roc_auc"][i] = np.nan

            try:
                dists["pr_auc"][i] = average_precision_score(yt, ys)
            except ValueError:
                dists["pr_auc"][i] = np.nan

    point = _classification_report_row(y_true, y_pred)

    if use_scores:

        try:
            point["roc_auc"] = roc_auc_score(y_true, y_score)
        except ValueError:
            point["roc_auc"] = float("nan")

        try:
            point["pr_auc"] = average_precision_score(y_true, y_score)
        except ValueError:
            point["pr_auc"] = float("nan")

    mean = {k: float(np.nanmean(dists[k])) for k in keys}
    std = {k: float(np.nanstd(dists[k], ddof=1)) for k in keys}
    ci_lo = {k: float(np.nanpercentile(dists[k], 2.5)) for k in keys}
    ci_hi = {k: float(np.nanpercentile(dists[k], 97.5)) for k in keys}

    return {
        "point": point,
        "dists": dists,
        "mean": mean,
        "std": std,
        "ci_lo": ci_lo,
        "ci_hi": ci_hi,
    }


def format_bootstrap_classification_report(
    boot_result,
    n_boot=N_BOOTSTRAP,
    digits=5,
):
    """
    Render the bootstrap result as a classification_report-shaped table,
    all values at `digits` decimal places, each cell as:
        point_estimate  (bootstrap_mean +/- bootstrap_std)  [95% CI: lo, hi]
    """

    point = boot_result["point"]
    mean = boot_result["mean"]
    std = boot_result["std"]
    ci_lo = boot_result["ci_lo"]
    ci_hi = boot_result["ci_hi"]

    def fmt(k):
        return (
            f"{point[k]:.{digits}f}  "
            f"({mean[k]:.{digits}f} +/- {std[k]:.{digits}f})  "
            f"[95% CI: {ci_lo[k]:.{digits}f}, {ci_hi[k]:.{digits}f}]"
        )

    lines = []

    lines.append(f"Bootstrap resamples: N = {n_boot}")

    lines.append(
        "Each cell = point_estimate_on_full_test_set  "
        "(bootstrap_mean +/- bootstrap_std)  [95% percentile CI]"
    )

    lines.append("-" * 100)

    header = (
        f"{'':<16}{'precision':<40}{'recall':<40}"
        f"{'f1-score':<40}{'support':<20}"
    )

    lines.append(header)

    lines.append(
        f"{'Not offensive':<16}"
        f"{fmt('precision_0'):<40}"
        f"{fmt('recall_0'):<40}"
        f"{fmt('f1_0'):<40}"
        f"{fmt('support_0'):<20}"
    )

    lines.append(
        f"{'Offensive':<16}"
        f"{fmt('precision_1'):<40}"
        f"{fmt('recall_1'):<40}"
        f"{fmt('f1_1'):<40}"
        f"{fmt('support_1'):<20}"
    )

    lines.append("")

    lines.append(f"{'accuracy':<16}{fmt('accuracy')}")

    lines.append(
        f"{'macro avg':<16}"
        f"{fmt('macro_precision'):<40}"
        f"{fmt('macro_recall'):<40}"
        f"{fmt('macro_f1'):<40}"
    )

    lines.append(
        f"{'weighted avg':<16}"
        f"{fmt('weighted_precision'):<40}"
        f"{fmt('weighted_recall'):<40}"
        f"{fmt('weighted_f1'):<40}"
    )

    if "roc_auc" in point:

        lines.append("")
        lines.append(f"{'ROC-AUC':<16}{fmt('roc_auc')}")
        lines.append(f"{'PR-AUC':<16}{fmt('pr_auc')}")

    return "\n".join(lines)


In [13]:
# ============================================================================
# BOOTSTRAP CONFIDENCE INTERVALS ON TEST SET
# ============================================================================
# Reuses test_labels / test_predictions / test_probabilities computed above
# -- no extra forward passes through the model.

print("\n" + "=" * 70)
print(f"BOOTSTRAP RESAMPLING ON TEST SET (N = {N_BOOTSTRAP})")
print("=" * 70)

test_boot_result = bootstrap_classification_report(
    test_labels,
    test_predictions,
    y_score=test_probabilities,
    n_boot=N_BOOTSTRAP,
    seed=BOOTSTRAP_SEED,
)

test_boot_report_str = format_bootstrap_classification_report(
    test_boot_result,
    n_boot=N_BOOTSTRAP,
    digits=5,
)

print(
    "\nBootstrap classification report "
    "(point | mean +/- std | 95% CI), 5 digits:"
)

print(test_boot_report_str)

print("\n" + "-" * 70)

print(
    "Bootstrap metrics, one per line "
    "(point | mean +/- std | 95% CI), 5 digits:"
)

_boot_label_map = {
    "precision_0": "Not offensive precision",
    "recall_0": "Not offensive recall",
    "f1_0": "Not offensive f1-score",
    "support_0": "Not offensive support",
    "precision_1": "Offensive precision",
    "recall_1": "Offensive recall",
    "f1_1": "Offensive f1-score",
    "support_1": "Offensive support",
    "accuracy": "Accuracy",
    "macro_precision": "Macro avg precision",
    "macro_recall": "Macro avg recall",
    "macro_f1": "Macro avg f1-score",
    "weighted_precision": "Weighted avg precision",
    "weighted_recall": "Weighted avg recall",
    "weighted_f1": "Weighted avg f1-score",
    "roc_auc": "ROC-AUC",
    "pr_auc": "PR-AUC",
}

for _key, _label in _boot_label_map.items():

    if _key not in test_boot_result["point"]:
        continue

    _pt = test_boot_result["point"][_key]
    _mn = test_boot_result["mean"][_key]
    _sd = test_boot_result["std"][_key]
    _lo = test_boot_result["ci_lo"][_key]
    _hi = test_boot_result["ci_hi"][_key]

    print(
        f"{_label:<26}: point={_pt:.5f} | "
        f"mean +/- std = {_mn:.5f} +/- {_sd:.5f} | "
        f"95% CI = [{_lo:.5f}, {_hi:.5f}]"
    )

bootstrap_report_path = os.path.join(SAVE_DIR, "bootstrap_report.txt")

with open(bootstrap_report_path, "w") as f:
    f.write(test_boot_report_str)

print("\nBootstrap report saved to:", bootstrap_report_path)



BOOTSTRAP RESAMPLING ON TEST SET (N = 10000)

Bootstrap classification report (point | mean +/- std | 95% CI), 5 digits:
Bootstrap resamples: N = 10000
Each cell = point_estimate_on_full_test_set  (bootstrap_mean +/- bootstrap_std)  [95% percentile CI]
----------------------------------------------------------------------------------------------------
                precision                               recall                                  f1-score                                support             
Not offensive   0.96946  (0.96943 +/- 0.00513)  [95% CI: 0.95902, 0.97900]0.90620  (0.90626 +/- 0.00840)  [95% CI: 0.88971, 0.92227]0.93676  (0.93675 +/- 0.00521)  [95% CI: 0.92635, 0.94676]1226.00000  (1226.04270 +/- 10.72614)  [95% CI: 1204.00000, 1247.00000]
Offensive       0.44175  (0.44168 +/- 0.03478)  [95% CI: 0.37430, 0.51000]0.72222  (0.72182 +/- 0.04047)  [95% CI: 0.64062, 0.79845]0.54819  (0.54727 +/- 0.03317)  [95% CI: 0.47973, 0.60997]126.00000  (125.95730 +/- 10.72614) 